In [2]:
import torch
import torch.nn as nn
from model import ResNet20
from data import train_loader, val_loader

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [4]:
model = ResNet20().to(device) #Build and move model's weight to GPU
loss_fn = nn.CrossEntropyLoss() #Cross entropyLoss function
optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4) 
EPOCHS = 100
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS) #lr 0.1 to 0 along a cosine curve


In [5]:
def train_one_epoch():
    model.train() #Training mode
    for images, labels in train_loader: #128 images at a time, about 352 batches
        images = images.to(device) #Move batch to GPU
        labels = labels.to(device) 
        outputs = model(images) #Forward pass
        loss = loss_fn(outputs, labels) #Loss
        optimizer.zero_grad() #Clear old grad
        loss.backward() #Backpropogation
        optimizer.step() #update weights

In [ ]:
def evaluate(loader): #Accuracy on any loader
    model.eval() #test mode
    correct = 0 
    total = 0
    with torch.no_grad(): #No grad needed
        for images, labels in loader: 
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images) #[256, 10]
            preds = outputs.argmax(dim=1) #Pick highest score
            correct += (preds == labels).sum().item() #Count matches
            total += labels.size(0) #Images in this batch
        return correct / total

In [7]:
print(evaluate(val_loader))

0.1116


In [8]:
import time, os
os.makedirs('../checkpoints', exist_ok=True)

best_acc = 0
for epoch in range(EPOCHS): 
    start = time.time()
    train_one_epoch()
    scheduler.step() #Move lr to next point on cosine curve
    val_acc = evaluate(val_loader)
    if val_acc > best_acc: #If new best, Save it.
        best_acc = val_acc
        torch.save(model.state_dict(), '../checkpoints/resnet20_best.pt')
    print(f"epoch {epoch+1}  val {val_acc:.4f}  best {best_acc:.4f}  lr {scheduler.get_last_lr()[0]:.4f}  time {time.time()-start:.0f}s")

epoch 1  val 0.4730  best 0.4730  lr 0.1000  time 11s
epoch 2  val 0.4466  best 0.4730  lr 0.0999  time 11s
epoch 3  val 0.6116  best 0.6116  lr 0.0998  time 11s
epoch 4  val 0.6418  best 0.6418  lr 0.0996  time 11s
epoch 5  val 0.7532  best 0.7532  lr 0.0994  time 12s
epoch 6  val 0.6874  best 0.7532  lr 0.0991  time 12s
epoch 7  val 0.6758  best 0.7532  lr 0.0988  time 11s
epoch 8  val 0.7182  best 0.7532  lr 0.0984  time 11s
epoch 9  val 0.7184  best 0.7532  lr 0.0980  time 11s
epoch 10  val 0.7834  best 0.7834  lr 0.0976  time 10s
epoch 11  val 0.7660  best 0.7834  lr 0.0970  time 11s
epoch 12  val 0.7628  best 0.7834  lr 0.0965  time 11s
epoch 13  val 0.7720  best 0.7834  lr 0.0959  time 11s
epoch 14  val 0.7984  best 0.7984  lr 0.0952  time 11s
epoch 15  val 0.7886  best 0.7984  lr 0.0946  time 11s
epoch 16  val 0.7512  best 0.7984  lr 0.0938  time 11s
epoch 17  val 0.6962  best 0.7984  lr 0.0930  time 11s
epoch 18  val 0.7552  best 0.7984  lr 0.0922  time 12s
epoch 19  val 0.772

In [9]:
from data import test_loader
model.load_state_dict(torch.load('../checkpoints/resnet20_best.pt')) #Put best epoch's weight back
test_acc = evaluate(test_loader)
print(f"test {test_acc : .4f}")

test  0.9144
